# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [22]:
from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [23]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 9


Root proiect: d:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [24]:
student_id = "Student_05"
handle = "RomaniaTVOFICIAL"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

d:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2\data\raw\Student_05_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [25]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': '5Zo8z-GdfIySq62DgIY7Rd83AdU',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '-0r5z3j3EyO6jlYD3EY3h5WmAjk',
   'id': 'UC5Bb0itu0pB46xykW32ck0g'}]}

In [26]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UC5Bb0itu0pB46xykW32ck0g'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [27]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'k6KM_UpNPgkPwqf8AcyjeyEpwFM',
 'id': {'kind': 'youtube#video', 'videoId': 'xMaix8nlvKM'},
 'snippet': {'publishedAt': '2026-05-08T20:38:12Z',
  'channelId': 'UC5Bb0itu0pB46xykW32ck0g',
  'title': 'Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte',
  'description': 'Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte: „Faptul că ai obținut un scor bun la olimpiade ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/xMaix8nlvKM/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/xMaix8nlvKM/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/xMaix8nlvKM/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Romania TV',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-08T20:38:12Z'}}

In [28]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'xMaix8nlvKM',
  'video_title': 'Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte',
  'video_date': '2026-05-08'},
 {'video_id': 'dMKV80dlYwg',
  'video_title': 'Boți, tiriboți și sprâncenoboți s-au activat pentru salvarea lui Bolojan.',
  'video_date': '2026-05-08'},
 {'video_id': 'ONHktCBpS3g',
  'video_title': 'Mii de curse aeriene urmează să fie anulate. Penuria de kerosen va schimba complet vacanțele',
  'video_date': '2026-05-08'},
 {'video_id': 'Y2iw0gD8UY0',
  'video_title': 'Un nou caz „Colectiv” în Mexic. Cinci oameni au murit după un incendiu devastator la concert',
  'video_date': '2026-05-08'},
 {'video_id': '2BVNXKA6BTI',
  'video_title': 'ANAT cere vouchere de vacanţă şi pentru mediul privat. Valoarea tichetelor, redusă la jumătate',
  'video_date': '2026-05-08'},
 {'video_id': 'HwfgQARrAYw',
  'video_title': 'SUA și Iranul se joacă de-a armistițiul. Trump: ”I-am spulberat”',
  'video_date': '2026-05-08'},
 {'video_id': 'jywz

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [29]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte


C:\Users\crist\AppData\Local\Temp\ipykernel_9680\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Boți, tiriboți și sprâncenoboți s-au activat pentru salvarea lui Bolojan.
Colectez: Mii de curse aeriene urmează să fie anulate. Penuria de kerosen va schimba compl
Colectez: Un nou caz „Colectiv” în Mexic. Cinci oameni au murit după un incendiu devastato
Colectez: ANAT cere vouchere de vacanţă şi pentru mediul privat. Valoarea tichetelor, redu
Colectez: SUA și Iranul se joacă de-a armistițiul. Trump: ”I-am spulberat”
Colectez: Un bărbat s-a aruncat cu parașuta de pe un bloc cu 15 etaje, în Râmnicu Vâlcea P
Colectez: Luare de ostatici într-o filială de bancă din Germania, intervenție masivă a pol
Colectez: Raportul care dă fiori. Peste jumătate dintre angajaţii statului ies la pensie î
Colectez: Se suspendă reducerea numărului de posturi din primării! Bolojan stabilise tăier


42

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [30]:
comments[:3]

[{'id': 'yt_xMaix8nlvKM_UgycrI_8O0noiDK6LXt4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'RomaniaTVOFICIAL',
  'text_raw': 'Kovesi  vrea ca Mucusor sa fie asasinat   ai doreste sa aiba un somn de veci , un somn profund. Justitia din Romania sa ia in serios aceasta expresie  ca si cum il amenita cu  moartea prin asasinare si trebue  condamnata  la ani grei de puscarie daca mai exista justitie corecta in Romania .',
  'video_id': 'xMaix8nlvKM',
  'video_title': 'Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte',
  'video_date': '2026-05-08',
  'comment_date': '2026-05-09',
  'likes': 2,
  'collected_at': '2026-05-10'},
 {'id': 'yt_xMaix8nlvKM_Ugz3pLyDRxG4zlAG_5N4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'RomaniaTVOFICIAL',
  'text_raw': 'Cînd te bagi să schimbi cursul firesc al lucrurilor, te anulezi singur! Acum încaieră-i drace, (că) și mie îmi place!',
  'video_id': 'xMaix8nlvKM',
  'video_title': 'Război între Kovesi 

In [31]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [32]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [33]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_xMaix8nlvKM_UgycrI_8O0noiDK6LXt4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'RomaniaTVOFICIAL',
 'text_raw': 'Kovesi  vrea ca Mucusor sa fie asasinat   ai doreste sa aiba un somn de veci , un somn profund. Justitia din Romania sa ia in serios aceasta expresie  ca si cum il amenita cu  moartea prin asasinare si trebue  condamnata  la ani grei de puscarie daca mai exista justitie corecta in Romania .',
 'video_id': 'xMaix8nlvKM',
 'video_title': 'Război între Kovesi și Nicușor Dan. Șefa EPPO îl trimite la „somn” pe președinte',
 'video_date': '2026-05-08',
 'comment_date': '2026-05-09',
 'likes': 2,
 'collected_at': '2026-05-10',
 'text': 'Kovesi vrea ca Mucusor sa fie asasinat ai doreste sa aiba un somn de veci , un somn profund. Justitia din Romania sa ia in serios aceasta expresie ca si cum il amenita cu moartea prin asasinare si trebue condamnata la ani grei de puscarie daca mai exista justitie corecta in Romania .'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [34]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 42
Comentarii după filtrarea lungimii: 25


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [35]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 25


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [36]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 25


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [37]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 25
Fișier: d:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2\data\cleaned\Student_05_youtube_clean.jsonl


# Functia de curatare

In [38]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [39]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 42
Comentarii curate: 25


In [40]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Kovesi  vrea ca Mucusor sa fie asasinat   ai doreste sa aiba un somn de veci , un somn profund. Justitia din Romania sa ia in serios aceasta expresie  ca si cum il amenita cu  moartea prin asasinare si trebue  condamnata  la ani grei de puscarie daca mai exista justitie corecta in Romania .
CLEAN: Kovesi vrea ca Mucusor sa fie asasinat ai doreste sa aiba un somn de veci , un somn profund. Justitia din Romania sa ia in serios aceasta expresie ca si cum il amenita cu moartea prin asasinare si trebue condamnata la ani grei de puscarie daca mai exista justitie corecta in Romania .
---
RAW: Cînd te bagi să schimbi cursul firesc al lucrurilor, te anulezi singur! Acum încaieră-i drace, (că) și mie îmi place!
CLEAN: Cînd te bagi să schimbi cursul firesc al lucrurilor, te anulezi singur! Acum încaieră-i drace, (că) și mie îmi place!
---
RAW: Lui kiovesi îi scade mandatul anul ăsta și se pare că așteaptă să se pregătească un nou post pentru ea în funcție mare în România și o încurcă Nicușor

In [41]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: d:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2\data\cleaned\Student_05_youtube_clean.jsonl
Comentarii salvate: 25


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.